# 1. Look at the data
Profile of `data/raw_applications.csv` for the brief's first point.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = next(
    root / "data" / "raw_applications.csv"
    for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "data" / "raw_applications.csv").exists()
)
df = pd.read_csv(DATA_PATH, keep_default_na=False)
missing_markers = {"", "null", "na", "n/a"}
missing = df.apply(
    lambda column: column.astype("string").str.strip().str.lower().isin(missing_markers)
)

print(f"{len(df):,} rows x {df.shape[1]} columns")
display(pd.DataFrame({"type": df.dtypes.astype(str), "missing": missing.sum()}))
display(df.drop(columns="tax_id").head())

In [ ]:
print(f"Repeated application IDs: {df['application_id'].duplicated().sum()}")

numeric_cols = ["employees", "annual_revenue", "annual_ebitda", "requested_amount"]
numeric = df[numeric_cols].apply(
    lambda column: pd.to_numeric(column.mask(missing[column.name]), errors="coerce")
)
profile = numeric.agg(["count", "min", "median", "max"]).T
profile["missing_or_placeholder"] = missing[numeric_cols].sum()
profile["invalid"] = (~missing[numeric_cols] & numeric.isna()).sum()
profile["negative"] = numeric.lt(0).sum()
profile["zero"] = numeric.eq(0).sum()
profile.loc["employees", "non_integer"] = int((numeric["employees"].dropna() % 1 != 0).sum())
display(profile)

dates = df.loc[~missing["incorporation_date"], "incorporation_date"].astype(str)
non_iso_dates = dates[~dates.str.fullmatch(r"\d{4}-\d{2}-\d{2}")]
parsed_dates = pd.to_datetime(dates, format="mixed", errors="coerce", dayfirst=True)
print(f"Non-ISO dates: {len(non_iso_dates)} / {len(dates)}; unparseable dates: {parsed_dates.isna().sum()}")
display(non_iso_dates.value_counts().head())

In [ ]:
for column in ["sector", "region", "source_system"]:
    print(f"\n{column} ({df[column].nunique()} values)")
    display(df[column].value_counts(dropna=False))

## Findings

The file has 196 rows and 12 columns. Four records repeat application IDs. There are 10 missing dates, 22 missing employee counts, 101 missing/placeholder EBITDA values, and 87 missing notes. Employee counts include one negative and two zero values; EBITDA has 10 negatives. Of 186 supplied dates, 30 are not ISO-formatted. Sector labels vary across 20 values, and the region list includes both `Cataluna` and `Catalunya`. The preview omits `tax_id`, which should be handled as sensitive data.

Ask the source team to confirm ID uniqueness, missing-value meanings, date and category standards, and whether repeated IDs represent revisions or duplicate submissions.

# 2. Decide how to store this

Keep each delivered CSV in a versioned, encrypted S3 bucket as the raw record. Never overwrite a delivery; it gives us a source snapshot to investigate or replay.

Use Postgres for the API's operational data: an application record and a separate score record for each scoring run. Store the submitted values, decision, risk factors, rule version, timestamp, and request ID so a decision can be explained later. Don't key only on `application_id` yet: four IDs repeat in this export, and the source needs to clarify whether those rows are duplicates or revisions. Keep the source ID, add an internal application ID, and define deduplication after that clarification.

Keep `tax_id` out of logs and restrict access to it. S3 suits durable source files; Postgres suits transactional updates and querying. The API should score the request body, not reread the CSV.

# 3. Serve a real request

The prototype was a batch script: it read the CSV and printed one decision per row. The new FastAPI endpoint scores one submitted application at a time; it does not read the export as part of a request.

`POST /score` accepts the fields this heuristic needs:

```json
{"application_id": "A-123", "employees": 10, "annual_revenue": 1000000, "requested_amount": 100000}
```

It returns the decision, the rule(s) that triggered it, the ruleset version and a request ID (also sent as `X-Request-ID`):

```json
{"application_id": "A-123", "decision": "low_risk", "risk_factors": [], "ruleset_version": "2026-10-v1", "request_id": "..."}
```

The formula and thresholds are unchanged. The request model requires a non-empty ID and positive employee, revenue, and request values; unknown fields are rejected. Invalid requests receive HTTP 422 rather than an error-shaped score. This prevents zero denominators and avoids guessing missing values, which matters given the gaps and invalid employee counts in the export.

Run locally from the repository root with `python -m uvicorn prototype.risk_scoring_prototype:app --reload`, then open `/docs` to try the endpoint. Callers authenticate with an `X-API-Key` header (a stopgap). Each request is saved to Postgres (SQLite locally) before the decision is returned, with the schema managed by Alembic migrations; if the write fails the service returns 503. The scorer itself stays free of HTTP and storage concerns. It was verified in Docker against Postgres 16.

# 4. Deploy and operate

**Run it.** Package the API in Docker, publish the image to ECR, and run it on ECS Fargate behind an authenticated HTTPS endpoint. Keep RDS Postgres private. The current endpoint is a working slice, not ready for production yet: it has no authentication, persistence, or rule version in its response.

**Change rules safely.** Give every ruleset an explicit version and record it with each score, alongside the image version. Replay historical applications and review which decisions change before release. Start with shadow traffic, then a small canary; keep the previous version available and roll back on elevated errors or unexpected score shifts. Keep database changes backward-compatible during rollout.

**Watch it before outcomes arrive.** Track request volume, latency, 4xx/5xx errors, validation rejections, score and risk-factor rates, and shifts in input distributions. Break these down by source and relevant applicant groups. Alert on meaningful changes against a baseline. These checks catch operational issues and drift; they do not prove the heuristic predicts repayment. Measure that once outcomes mature.

**Make decisions debuggable.** Store the validated scoring inputs, result, risk factors, timestamp, request ID, and ruleset version with the score in restricted Postgres. Keep logs structured and include request ID, internal application ID, ruleset version, status, and latency; never log `tax_id` or the full financial payload.

When Data Science updates the formula, keep the endpoint contract stable, test the new version, replay the same eligible applications and inspect decision changes, then shadow/canary it. Roll back by restoring the previous image and ruleset. The scoring code is intentionally separate from HTTP validation so a rule change does not require redesigning the API.